In [ ]:
from diffusers import StableDiffusionPipeline
import torch

# 假设你已经训练好了 LoRA 层
# unet_lora_layers 和 text_encoder_lora_layers 是训练得到的 LoRA 层

# 创建 pipeline（用于调用 save_lora_weights）
pipeline = StableDiffusionPipeline.from_pretrained(
    "base-model-path",
    torch_dtype=torch.float16
).to("cuda")

# 保存 LoRA 权重（只保存 LoRA 参数，不保存基础模型）
pipeline.save_lora_weights(
    save_directory="./lora_weights",
    unet_lora_layers=unet_lora_layers,           # UNet 的 LoRA 层
    text_encoder_lora_layers=text_encoder_lora_layers,  # 文本编码器的 LoRA 层（可选）
    weight_name="pytorch_lora_weights.safetensors",  # 文件名
    safe_serialization=True  # 使用 safetensors 格式（推荐）
)

In [4]:
import torch
from diffusers import (
    AutoencoderKL,
    DDPMScheduler,
    StableDiffusionControlNetPipeline,
    UNet2DConditionModel,
    UniPCMultistepScheduler,
)
from model_with_camera import NewUNet2DConditionModel
temp_unet = UNet2DConditionModel.from_pretrained(
"stable-diffusion-v1-5/stable-diffusion-v1-5", subfolder="unet", revision=None, variant=None
)
unet = NewUNet2DConditionModel(**temp_unet.config, camera_dim=16)
state_dict = torch.load("/home/viplab/liukejia/normal-depth-diffusion/diffusers_nd/py/nd_mv_unet.pth", map_location="cpu")
unet.load_state_dict(state_dict, strict=True)
unet.state_dict().keys()

odict_keys(['conv_in.weight', 'conv_in.bias', 'time_embedding.linear_1.weight', 'time_embedding.linear_1.bias', 'time_embedding.linear_2.weight', 'time_embedding.linear_2.bias', 'down_blocks.0.attentions.0.norm.weight', 'down_blocks.0.attentions.0.norm.bias', 'down_blocks.0.attentions.0.proj_in.weight', 'down_blocks.0.attentions.0.proj_in.bias', 'down_blocks.0.attentions.0.transformer_blocks.0.norm1.weight', 'down_blocks.0.attentions.0.transformer_blocks.0.norm1.bias', 'down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_q.weight', 'down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_k.weight', 'down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_v.weight', 'down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_out.0.weight', 'down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_out.0.bias', 'down_blocks.0.attentions.0.transformer_blocks.0.norm2.weight', 'down_blocks.0.attentions.0.transformer_blocks.0.norm2.bias', 'down_blocks.0.attentions.0.transformer_blocks.0.

In [ ]:
old_keys = list(unet.state_dict().keys())

from peft import LoraConfig, inject_adapter_in_model

unet_lora_config = LoraConfig(r=8, lora_alpha=8, init_lora_weights="gaussian", target_modules=["to_k_proj", "to_q_proj", "to_v_proj", "to_out.0"])
unet = inject_adapter_in_model(unet_lora_config, unet, adapter_name="unet_lora")
print("Injected UNet LoRA adapters")

new_keys = list(unet.state_dict().keys())

lora_keys = []

for k in new_keys:
    if k not in old_keys:
        print(f"New key: {k}")
        lora_keys.append(k)


Injected UNet LoRA adapters
New key: down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_out.0.base_layer.weight
New key: down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_out.0.base_layer.bias
New key: down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_out.0.lora_A.unet_lora.weight
New key: down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_out.0.lora_B.unet_lora.weight
New key: down_blocks.0.attentions.0.transformer_blocks.0.attn2.to_out.0.base_layer.weight
New key: down_blocks.0.attentions.0.transformer_blocks.0.attn2.to_out.0.base_layer.bias
New key: down_blocks.0.attentions.0.transformer_blocks.0.attn2.to_out.0.lora_A.unet_lora.weight
New key: down_blocks.0.attentions.0.transformer_blocks.0.attn2.to_out.0.lora_B.unet_lora.weight
New key: down_blocks.0.attentions.1.transformer_blocks.0.attn1.to_out.0.base_layer.weight
New key: down_blocks.0.attentions.1.transformer_blocks.0.attn1.to_out.0.base_layer.bias
New key: down_blocks.0.attentions.1.transformer_blocks